# Document-Level Classification: Model Comparison

This notebook systematically compares multiple transformer backbones for the document-level fact vs opinion classification task.

**Key Features:**
- Multiple models tested (BERT-tiny, BERT-base, RoBERTa, DistilBERT)
- Fixed seeds and hyperparameters for reproducibility
- SVM + RNN evaluation for each model
- Results table + visualizations
- Confusion matrices saved to disk

In [11]:
import sys
import json
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple
import random

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report
from sklearn import svm
from tqdm.auto import tqdm

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

print(f"✓ All imports successful")

✓ All imports successful


## 1. Setup & Utilities

In [12]:
# Find repo root
def find_repo_root():
    current = Path.cwd()
    while current != current.parent:
        if (current / "src").exists() and (current / "README.md").exists():
            return current
        current = current.parent
    raise RuntimeError("Could not find repo root")

REPO_ROOT = find_repo_root()
SRC_ROOT = REPO_ROOT / "src"
DATA_DIR = SRC_ROOT / "data"
CHECKPOINT_DIR = SRC_ROOT / "checkpoints"
OUTPUT_DIR = REPO_ROOT / "outputs" / "experiments"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Add src to path so we can import mcc and doc_clf
sys.path.insert(0, str(SRC_ROOT))

from mcc.data import MCC_LABELS, LABEL2ID as MCC_LABEL2ID
from mcc.models import BertMCCClassifier, MCCModelConfig
from doc_clf.features import extract_features_for_corpus, load_documents
from doc_clf.models import ArgumentRNNClassifier, ArgumentRNNConfig, compute_macro_f1, train_svm
from doc_clf.train import SequenceDataset, collate_sequences

print(f"✓ Repo root: {REPO_ROOT}")
print(f"✓ Output directory: {OUTPUT_DIR}")

✓ Repo root: c:\Users\eliraz.eliezer.madar\Desktop\Sem-Project\sem-project
✓ Output directory: c:\Users\eliraz.eliezer.madar\Desktop\Sem-Project\sem-project\outputs\experiments


In [13]:
# Global settings
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

def set_seed(seed: int = SEED):
    """Set seed for reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

print(f"🔧 Device: {DEVICE}")
print(f"🌱 Seed: {SEED}")
print(f"⏱️  Timestamp: {TIMESTAMP}")

🔧 Device: cpu
🌱 Seed: 42
⏱️  Timestamp: 20260124_092302


## 2. Data Loading & Preprocessing

## Data Source: Real Dataset

**Status**: ✅ Now using the real dataset from `dataset.csv` (converted to JSONL format)
- **Original file**: `dataset.csv` (4,426 articles)
- **Conversion**: Automatically split into 80/20 train/test (3,540 training, 886 test)
- **Labels**: Balanced between "fact" (47.1% train, 49.1% test) and "opinion" (52.9% train, 50.9% test)
- **Date converted**: January 20, 2026

In [14]:
def load_jsonl(path: Path) -> List[dict]:
    """Load JSONL file."""
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def label_to_int(label: str) -> int:
    """Convert label string to binary int (fact=0, opinion=1)."""
    return 1 if label.lower() == "opinion" else 0

# Load data from converted dataset.csv (now in JSONL format)
print("📥 Loading data from dataset.csv (converted to JSONL)...")
train_data = load_jsonl(DATA_DIR / "news_train.jsonl")
test_data = load_jsonl(DATA_DIR / "news_test.jsonl")

# For compatibility, ensure 'text' field exists in data
for doc in train_data + test_data:
    if 'text' not in doc and 'article_content' in doc:
        doc['text'] = doc['article_content']

train_labels = np.array([label_to_int(d["label"]) for d in train_data])
test_labels = np.array([label_to_int(d["label"]) for d in test_data])

print(f"✓ Training samples: {len(train_data)}")
print(f"✓ Test samples: {len(test_data)}")
print(f"\nLabel distribution (train): {np.bincount(train_labels)}")
print(f"Label distribution (test): {np.bincount(test_labels)}")
print(f"\n📊 Data source: dataset.csv (4,426 articles, 80/20 train/test split)")

📥 Loading data from dataset.csv (converted to JSONL)...
✓ Training samples: 3540
✓ Test samples: 886

Label distribution (train): [1669 1871]
Label distribution (test): [435 451]

📊 Data source: dataset.csv (4,426 articles, 80/20 train/test split)


## 3. MCC Model Configuration

First, we need to ensure the MCC checkpoint exists. If not, we train one with a small model.

In [15]:
# Per-model MCC checkpoint handling to avoid shape mismatches across backbones

def mcc_checkpoint_for(model_name: str) -> Path:
    """Return a checkpoint path dedicated to the given backbone."""
    safe_name = model_name.replace("/", "_")
    return CHECKPOINT_DIR / f"mcc_{safe_name}.pt"

def ensure_mcc_checkpoint(model_name: str) -> Path:
    """Create a minimal MCC checkpoint for the given backbone if missing."""
    ckpt_path = mcc_checkpoint_for(model_name)
    if ckpt_path.exists():
        print(f"✓ MCC checkpoint found for {model_name}: {ckpt_path}")
        return ckpt_path
    print(f"⚠ MCC checkpoint not found for {model_name}. Creating minimal checkpoint...")
    mcc_config = MCCModelConfig(pretrained_model_name=model_name, num_labels=3)
    mcc_model = BertMCCClassifier.from_config(mcc_config)
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    torch.save(mcc_model.state_dict(), ckpt_path)
    print(f"✓ Saved MCC checkpoint for {model_name} to: {ckpt_path}")
    return ckpt_path

# Initialize a default tiny checkpoint so the first run remains fast
DEFAULT_MCC_BACKBONE = "prajjwal1/bert-tiny"
mcc_checkpoint_path = ensure_mcc_checkpoint(DEFAULT_MCC_BACKBONE)


✓ MCC checkpoint found for prajjwal1/bert-tiny: c:\Users\eliraz.eliezer.madar\Desktop\Sem-Project\sem-project\src\checkpoints\mcc_prajjwal1_bert-tiny.pt


## 4. Define Models to Compare

In [16]:
# Training configuration for multi-epoch RNN training
RNN_EPOCHS = 15  # Number of epochs to train RNN per model
VAL_SPLIT = 0.2  # Validation set fraction (20% of training data)

print(f"⚙️  RNN Configuration:")
print(f"  Epochs per model: {RNN_EPOCHS}")
print(f"  Validation split: {VAL_SPLIT * 100:.0f}%")
print(f"  Train/val from: {int((1 - VAL_SPLIT) * 3540)} / {int(VAL_SPLIT * 3540)} samples")


⚙️  RNN Configuration:
  Epochs per model: 15
  Validation split: 20%
  Train/val from: 2832 / 708 samples


## 5. SVM + RNN Experiment Runner

In [17]:
def train_rnn_epoch_with_eval(train_loader, val_loader, model, optimizer, device, epoch):
    """
    Train RNN for one epoch and evaluate on validation set.
    Returns (train_loss, val_f1_macro).
    """
    criterion = torch.nn.CrossEntropyLoss()
    
    # Training phase
    model.train()
    total_loss = 0.0
    for sequences, lengths, labels in tqdm(train_loader, desc=f"RNN train (epoch {epoch+1})", leave=False):
        sequences = sequences.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(sequences, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    train_loss = total_loss / max(len(train_loader), 1)
    
    # Validation phase
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for sequences, lengths, labels in tqdm(val_loader, desc=f"RNN eval (epoch {epoch+1})", leave=False):
            sequences = sequences.to(device)
            logits = model(sequences, lengths)
            preds = logits.argmax(dim=-1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.tolist())
    
    val_f1 = f1_score(all_labels, all_preds, average="macro")
    return train_loss, val_f1


def get_rnn_predictions(loader, model, device):
    """
    Evaluate RNN on a loader and return predictions and labels.
    """
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for sequences, lengths, labels in tqdm(loader, desc="RNN eval", leave=False):
            sequences = sequences.to(device)
            logits = model(sequences, lengths)
            preds = logits.argmax(dim=-1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.tolist())
    return np.array(all_preds), np.array(all_labels)


print("✓ Multi-epoch training utilities with validation defined")


✓ Multi-epoch training utilities with validation defined


In [18]:
def run_experiment(model_name: str, train_labels: np.ndarray, test_labels: np.ndarray) -> Dict:
    """
    Run a complete experiment for a given model with multi-epoch RNN training.
    Saves best RNN checkpoint per model based on validation F1.
    Returns a dict with SVM and best RNN metrics.
    """
    print(f"\n{'='*70}")
    print(f"Model: {model_name}")
    print(f"{'='*70}")
    
    # Ensure we use a matching MCC checkpoint for this backbone
    model_mcc_ckpt = ensure_mcc_checkpoint(model_name)
    
    result = {"model": model_name}
    
    try:
        # ===== STAGE 1: Extract features using MCC model =====
        print(f"\n[1/5] Extracting MCC features...")
        
        ratio_features, sequences, _ = extract_features_for_corpus(
            DATA_DIR / "news_train.jsonl",
            model_mcc_ckpt,
            model_name=model_name,
            device=DEVICE,
        )
        
        ratio_features_test, sequences_test, _ = extract_features_for_corpus(
            DATA_DIR / "news_test.jsonl",
            model_mcc_ckpt,
            model_name=model_name,
            device=DEVICE,
        )
        
        # ===== STAGE 2: Train SVM =====
        print(f"[2/5] Training SVM classifier...")
        
        feature_matrix = np.array([
            [feat[f"ratio_{label}"] for label in MCC_LABELS]
            for feat in ratio_features
        ])
        feature_matrix_test = np.array([
            [feat[f"ratio_{label}"] for label in MCC_LABELS]
            for feat in ratio_features_test
        ])
        
        svm_model = svm.SVC(kernel="linear", random_state=SEED)
        svm_model.fit(feature_matrix, train_labels)
        
        svm_train_preds = svm_model.predict(feature_matrix)
        svm_test_preds = svm_model.predict(feature_matrix_test)
        
        result["svm_train_f1_macro"] = f1_score(train_labels, svm_train_preds, average="macro")
        result["svm_train_f1_weighted"] = f1_score(train_labels, svm_train_preds, average="weighted")
        result["svm_train_acc"] = accuracy_score(train_labels, svm_train_preds)
        
        result["svm_test_f1_macro"] = f1_score(test_labels, svm_test_preds, average="macro")
        result["svm_test_f1_weighted"] = f1_score(test_labels, svm_test_preds, average="weighted")
        result["svm_test_acc"] = accuracy_score(test_labels, svm_test_preds)
        result["svm_cm"] = confusion_matrix(test_labels, svm_test_preds)
        
        print(f"  Train F1: {result['svm_train_f1_macro']:.4f} | Test F1: {result['svm_test_f1_macro']:.4f}")
        
        # ===== STAGE 3: Prepare RNN data with train/val split =====
        print(f"\n[3/5] Preparing RNN datasets (train/val/test)...")
        
        # Create a validation split from training data
        n_train = len(sequences)
        val_size = int(n_train * VAL_SPLIT)
        
        indices = np.arange(n_train)
        np.random.seed(SEED)
        np.random.shuffle(indices)
        
        val_indices = indices[:val_size]
        train_indices = indices[val_size:]
        
        sequences_train = [sequences[i] for i in train_indices]
        labels_train = train_labels[train_indices]
        sequences_val = [sequences[i] for i in val_indices]
        labels_val = train_labels[val_indices]
        
        train_dataset = SequenceDataset(sequences_train, labels_train)
        val_dataset = SequenceDataset(sequences_val, labels_val)
        test_dataset = SequenceDataset(sequences_test, test_labels)
        
        train_loader = DataLoader(
            train_dataset, batch_size=4, shuffle=True, collate_fn=collate_sequences
        )
        val_loader = DataLoader(
            val_dataset, batch_size=4, collate_fn=collate_sequences
        )
        test_loader = DataLoader(
            test_dataset, batch_size=4, collate_fn=collate_sequences
        )
        
        # ===== STAGE 4: Multi-epoch RNN training with best checkpoint =====
        print(f"[4/5] Training RNN for {RNN_EPOCHS} epochs (saving best checkpoint)...")
        
        rnn_config = ArgumentRNNConfig(vocab_size=len(MCC_LABEL2ID))
        rnn_model = ArgumentRNNClassifier(rnn_config).to(DEVICE)
        rnn_optimizer = torch.optim.Adam(rnn_model.parameters(), lr=1e-3)
        
        # Per-model checkpoint for best RNN
        safe_name = model_name.replace("/", "_")
        best_rnn_ckpt = CHECKPOINT_DIR / f"best_rnn_{safe_name}.pt"
        
        best_val_f1 = -1.0
        epoch_history = []
        
        for epoch in range(RNN_EPOCHS):
            train_loss, val_f1 = train_rnn_epoch_with_eval(
                train_loader, val_loader, rnn_model, rnn_optimizer, DEVICE, epoch
            )
            epoch_history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_f1": val_f1})
            
            print(f"  Epoch {epoch + 1}/{RNN_EPOCHS}: Loss={train_loss:.4f}, Val F1={val_f1:.4f}", end="")
            
            # Save best model
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                torch.save(rnn_model.state_dict(), best_rnn_ckpt)
                print(f" ✓ (new best)")
            else:
                print()
        
        result["epoch_history"] = epoch_history
        print(f"  Best validation F1: {best_val_f1:.4f} (saved to {best_rnn_ckpt.name})")
        
        # ===== STAGE 5: Evaluate best RNN checkpoint =====
        print(f"\n[5/5] Evaluating best RNN checkpoint on test set...")
        rnn_model.load_state_dict(torch.load(best_rnn_ckpt))
        
        rnn_train_preds, rnn_train_labels = get_rnn_predictions(train_loader, rnn_model, DEVICE)
        rnn_test_preds, rnn_test_labels = get_rnn_predictions(test_loader, rnn_model, DEVICE)
        
        result["rnn_train_f1_macro"] = f1_score(rnn_train_labels, rnn_train_preds, average="macro")
        result["rnn_train_f1_weighted"] = f1_score(rnn_train_labels, rnn_train_preds, average="weighted")
        result["rnn_train_acc"] = accuracy_score(rnn_train_labels, rnn_train_preds)
        
        result["rnn_test_f1_macro"] = f1_score(rnn_test_labels, rnn_test_preds, average="macro")
        result["rnn_test_f1_weighted"] = f1_score(rnn_test_labels, rnn_test_preds, average="weighted")
        result["rnn_test_acc"] = accuracy_score(rnn_test_labels, rnn_test_preds)
        result["rnn_cm"] = confusion_matrix(rnn_test_labels, rnn_test_preds)
        result["best_rnn_checkpoint"] = str(best_rnn_ckpt)
        result["best_val_f1"] = best_val_f1
        
        print(f"  Train F1: {result['rnn_train_f1_macro']:.4f} | Test F1: {result['rnn_test_f1_macro']:.4f}")
        
        result["status"] = "success"
        print(f"\n✓ Experiment completed")
        
    except Exception as e:
        print(f"\n✗ Error: {e}")
        import traceback
        traceback.print_exc()
        result["status"] = "failed"
        result["error"] = str(e)
    
    return result

print("✓ Experiment runner defined (complete with all 5 stages)")

✓ Experiment runner defined (complete with all 5 stages)


## 6. Run All Experiments

In [19]:
# Define models to compare
MODELS_TO_TEST = [
    "prajjwal1/bert-tiny",
    "bert-base-uncased",
    "distilbert-base-uncased",
    "roberta-base",
]
print(f"\n{'#'*70}")
print(f"# DOCUMENT-LEVEL CLASSIFICATION: MULTI-MODEL COMPARISON")
print(f"#")
print(f"# Seed: {SEED}")
print(f"# Device: {DEVICE}")
print(f"# Models: {len(MODELS_TO_TEST)}")
print(f"{'#'*70}")

results = []

for i, model_name in enumerate(MODELS_TO_TEST, 1):
    print(f"\n[{i}/{len(MODELS_TO_TEST)}] Running experiment...")
    result = run_experiment(model_name, train_labels, test_labels)
    results.append(result)
    
    # Clear GPU cache between experiments
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n{'='*70}")
print(f"✓ All experiments completed")
print(f"{'='*70}")


######################################################################
# DOCUMENT-LEVEL CLASSIFICATION: MULTI-MODEL COMPARISON
#
# Seed: 42
# Device: cpu
# Models: 4
######################################################################

[1/4] Running experiment...

Model: prajjwal1/bert-tiny
✓ MCC checkpoint found for prajjwal1/bert-tiny: c:\Users\eliraz.eliezer.madar\Desktop\Sem-Project\sem-project\src\checkpoints\mcc_prajjwal1_bert-tiny.pt

[1/5] Extracting MCC features...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Processing news_train: 100%|██████████| 3540/3540 [01:40<00:00, 35.14doc/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Processing news_test: 100%|██████████| 886/886 [00:22<00:00, 39.87doc/s]


[2/5] Training SVM classifier...
  Train F1: 0.7958 | Test F1: 0.8020

[3/5] Preparing RNN datasets (train/val/test)...
[4/5] Training RNN for 15 epochs (saving best checkpoint)...


  Epoch 1/15: Loss=0.4259, Val F1=0.9011 ✓ (new best)


  Epoch 2/15: Loss=0.2695, Val F1=0.9162 ✓ (new best)


  Epoch 3/15: Loss=0.2542, Val F1=0.9122


... (63 outputs left out)

  Epoch 13/15: Loss=0.3326, Val F1=0.8798


  Epoch 14/15: Loss=0.3332, Val F1=0.8879


  Epoch 15/15: Loss=0.3321, Val F1=0.8879
  Best validation F1: 0.8879 (saved to best_rnn_roberta-base.pt)

[5/5] Evaluating best RNN checkpoint on test set...


  Train F1: 0.8742 | Test F1: 0.8677

✓ Experiment completed

✓ All experiments completed


## 7. Results Summary

In [23]:
# Extract successful results only
successful_results = [r for r in results if r["status"] == "success"]

# Create results dataframe for SVM
svm_results = []
for r in successful_results:
    svm_results.append({
        "Model": r["model"],
        "SVM Train F1 (macro)": f"{r['svm_train_f1_macro']:.4f}",
        "SVM Test F1 (macro)": f"{r['svm_test_f1_macro']:.4f}",
        "SVM Test F1 (weighted)": f"{r['svm_test_f1_weighted']:.4f}",
        "SVM Test Acc": f"{r['svm_test_acc']:.4f}",
    })

svm_df = pd.DataFrame(svm_results)
print(f"\n" + "="*70)
print(f"SVM CLASSIFIER RESULTS (sorted by Test F1)")
print(f"="*70)
print(svm_df.to_string(index=False))


SVM CLASSIFIER RESULTS (sorted by Test F1)
                  Model SVM Train F1 (macro) SVM Test F1 (macro) SVM Test F1 (weighted) SVM Test Acc
    prajjwal1/bert-tiny               0.3458              0.3373                 0.3434       0.5090
      bert-base-uncased               0.3458              0.3373                 0.3434       0.5090
distilbert-base-uncased               0.7705              0.7696                 0.7702       0.7743


In [24]:
# Create results dataframe for RNN
rnn_results = []
for r in successful_results:
    rnn_results.append({
        "Model": r["model"],
        "RNN Train F1 (macro)": f"{r['rnn_train_f1_macro']:.4f}",
        "RNN Test F1 (macro)": f"{r['rnn_test_f1_macro']:.4f}",
        "RNN Test F1 (weighted)": f"{r['rnn_test_f1_weighted']:.4f}",
        "RNN Test Acc": f"{r['rnn_test_acc']:.4f}",
    })

rnn_df = pd.DataFrame(rnn_results)
print(f"\n" + "="*70)
print(f"RNN CLASSIFIER RESULTS (sorted by Test F1)")
print(f"="*70)
print(rnn_df.to_string(index=False))


RNN CLASSIFIER RESULTS (sorted by Test F1)
                  Model RNN Train F1 (macro) RNN Test F1 (macro) RNN Test F1 (weighted) RNN Test Acc
    prajjwal1/bert-tiny               0.8767              0.8677                 0.8678       0.8679
      bert-base-uncased               0.8761              0.8688                 0.8689       0.8691
distilbert-base-uncased               0.8938              0.8911                 0.8913       0.8916


## 8. Save Results

In [ ]:
# Save results CSV
results_csv = OUTPUT_DIR / f"doc_clf_comparison_{TIMESTAMP}.csv"

export_data = []
for r in successful_results:
    export_data.append({
        "Model": r["model"],
        "SVM_Train_F1_Macro": r["svm_train_f1_macro"],
        "SVM_Test_F1_Macro": r["svm_test_f1_macro"],
        "SVM_Test_F1_Weighted": r["svm_test_f1_weighted"],
        "SVM_Test_Accuracy": r["svm_test_acc"],
        "RNN_Train_F1_Macro": r["rnn_train_f1_macro"],
        "RNN_Test_F1_Macro": r["rnn_test_f1_macro"],
        "RNN_Test_F1_Weighted": r["rnn_test_f1_weighted"],
        "RNN_Test_Accuracy": r["rnn_test_acc"],
    })

export_df = pd.DataFrame(export_data)
export_df.to_csv(results_csv, index=False)
print(f"✓ Results saved to: {results_csv}")

In [ ]:
# Save confusion matrices as images (text format)
import matplotlib.pyplot as plt
import seaborn as sns

for r in successful_results:
    if "svm_cm" in r:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        
        # SVM confusion matrix
        sns.heatmap(r["svm_cm"], annot=True, fmt="d", cmap="Blues", ax=axes[0], cbar=False)
        axes[0].set_title(f"SVM - {r['model']}")
        axes[0].set_ylabel("True Label")
        axes[0].set_xlabel("Predicted Label")
        axes[0].set_xticklabels(["Fact", "Opinion"])
        axes[0].set_yticklabels(["Fact", "Opinion"])
        
        # RNN confusion matrix
        sns.heatmap(r["rnn_cm"], annot=True, fmt="d", cmap="Greens", ax=axes[1], cbar=False)
        axes[1].set_title(f"RNN - {r['model']}")
        axes[1].set_ylabel("True Label")
        axes[1].set_xlabel("Predicted Label")
        axes[1].set_xticklabels(["Fact", "Opinion"])
        axes[1].set_yticklabels(["Fact", "Opinion"])
        
        plt.tight_layout()
        model_safe_name = r["model"].replace("/", "_")
        cm_path = OUTPUT_DIR / f"confusion_matrix_{model_safe_name}_{TIMESTAMP}.png"
        plt.savefig(cm_path, dpi=150, bbox_inches="tight")
        print(f"✓ Confusion matrix saved: {cm_path.name}")
        plt.close()

print(f"\n✓ All results saved to: {OUTPUT_DIR}")

## 9. Summary & Best Model

In [25]:
print(f"\n" + "="*70)
print(f"SUMMARY")
print(f"="*70)
print(f"\nExperiments run: {len(results)}")
print(f"Successful: {len(successful_results)}")
print(f"Failed: {len(results) - len(successful_results)}")

if successful_results:
    # Find best SVM and RNN models
    best_svm = max(successful_results, key=lambda x: x["svm_test_f1_macro"])
    best_rnn = max(successful_results, key=lambda x: x["rnn_test_f1_macro"])
    
    print(f"\n📊 Best Performers:")
    print(f"  Best SVM: {best_svm['model']}")
    print(f"    Test F1 (macro): {best_svm['svm_test_f1_macro']:.4f}")
    print(f"    Test Accuracy: {best_svm['svm_test_acc']:.4f}")
    
    print(f"\n  Best RNN: {best_rnn['model']}")
    print(f"    Test F1 (macro): {best_rnn['rnn_test_f1_macro']:.4f}")
    print(f"    Test Accuracy: {best_rnn['rnn_test_acc']:.4f}")

print(f"\n📁 Results location: {OUTPUT_DIR}")
print(f"⏱️  Timestamp: {TIMESTAMP}")
print(f"\n✓ Comparison complete!")


SUMMARY

Experiments run: 3
Successful: 3
Failed: 0

📊 Best Performers:
  Best SVM: distilbert-base-uncased
    Test F1 (macro): 0.7696
    Test Accuracy: 0.7743

  Best RNN: distilbert-base-uncased
    Test F1 (macro): 0.8911
    Test Accuracy: 0.8916

📁 Results location: c:\Users\elira\study\fifth\CLS\sem_code\sem-project\outputs\experiments
⏱️  Timestamp: 20260120_153436

✓ Comparison complete!


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# הגדרת סגנון גרפים נקי
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'DejaVu Sans' # תמיכה בסיסית בעברית אם צריך, לרוב עדיף אנגלית בגרפים טכניים

# =============================================
# נתונים (חולצו ידנית מהלוגים שסופקו)
# =============================================

models = ['bert-tiny', 'bert-base-uncased', 'distilbert-base-uncased', 'roberta-base']

# נתוני אימון ה-RNN לפי אפוקים (Validation F1)
rnn_epochs_data = {
    'bert-tiny': [0.9011, 0.9162, 0.9122, 0.9131, 0.8955, 0.9084, 0.9204, 0.9011, 0.9248, 0.8951, 0.9204, 0.9176, 0.9173, 0.9086, 0.9102],
    'bert-base-uncased': [0.8853, 0.8838, 0.8924, 0.8630, 0.8785, 0.8965, 0.8634, 0.8410, 0.8672, 0.8813, 0.8646, 0.8672, 0.8890, 0.8841, 0.8882],
    'distilbert-base-uncased': [0.8935, 0.9109, 0.8997, 0.9164, 0.9123, 0.9137, 0.9152, 0.9150, 0.9054, 0.9136, 0.9178, 0.9067, 0.8976, 0.8884, 0.9207],
    'roberta-base': [0.8728, 0.8757, 0.8876, 0.8729, 0.8876, 0.8757, 0.8757, 0.8793, 0.8879, 0.8728, 0.8757, 0.8757, 0.8798, 0.8879, 0.8879]
}

# תוצאות סופיות על סט הבדיקה (Test F1)
final_results_data = {
    'Model': models * 2,
    'Classifier': ['SVM'] * 4 + ['RNN (Best Checkpoint)'] * 4,
    'Test F1 Score': [
        0.8020, 0.5702, 0.6811, 0.3373,  # SVM results
        0.9141, 0.8586, 0.8825, 0.8677   # RNN Test results
    ]
}
df_final = pd.DataFrame(final_results_data)


# =============================================
# גרף 1: התקדמות אימון ה-RNN (לפי אפוקים)
# =============================================

plt.figure(figsize=(12, 6))

# יצירת DataFrame לנתוני האפוקים כדי להקל על הציור ב-Seaborn
epoch_list = []
for model, scores in rnn_epochs_data.items():
    for i, score in enumerate(scores):
        epoch_list.append({'Model': model, 'Epoch': i + 1, 'Validation F1': score})
df_epochs = pd.DataFrame(epoch_list)

# ציור גרף קו
sns.lineplot(data=df_epochs, x='Epoch', y='Validation F1', hue='Model', marker='o', linewidth=2.5)

# הוספת כותרות ותוויות
plt.title('RNN Classifier Training Progress (Validation F1 over 15 Epochs)', fontsize=16, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Validation F1 Score', fontsize=12)
plt.xticks(range(1, 16)) # לוודא שכל האפוקים מופיעים בציר X
plt.legend(title='Base Model', title_fontsize='12', fontsize='11')
plt.grid(True, linestyle='--')
plt.tight_layout()

# שמירת הגרף
plt.savefig('rnn_training_progress.png', dpi=300)
plt.show()


# =============================================
# גרף 2: השוואת התוצאות הסופיות הטובות ביותר (SVM vs RNN)
# =============================================

plt.figure(figsize=(12, 6))

# יצירת גרף עמודות מקובץ (Grouped Bar Chart)
barplot = sns.barplot(
    data=df_final,
    x='Model',
    y='Test F1 Score',
    hue='Classifier',
    palette='viridis'
)

# הוספת ערכים מספריים מעל העמודות
for container in barplot.containers:
    barplot.bar_label(container, fmt='%.3f', padding=3, fontsize=10)

# הוספת כותרות ותוויות
plt.title('Final Test Performance Comparison: SVM vs. RNN Classifier', fontsize=16, fontweight='bold')
plt.xlabel('Pre-trained Base Model', fontsize=12)
plt.ylabel('Test F1 Score', fontsize=12)
plt.ylim(0, 1.0) # קיבוע ציר Y בין 0 ל-1
plt.legend(title='Classifier Head', title_fontsize='12', fontsize='11', loc='upper right')
plt.grid(axis='y', linestyle='--')
plt.tight_layout()

# שמירת הגרף
plt.savefig('final_model_comparison.png', dpi=300)
plt.show()